# 2. Physiology checks and the measurement window

Checks on the last three beats in a viable region (acceptance.physiology): one activation per beat,
repolarisation inside the response window, quiet lesion core, planar wave along +x, stable latency,
amplitude and APD90, and conduction velocity within 2 % of 0.90 m/s. Engineering criteria, not
validated biological limits.

The measurement window is then derived from the intact last-beat activation of the top surface under
the grid (DL-2026-10-28-08), before any electrogram is read.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "final_model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v3_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
physiology = run.runStage("physiology")
pd.DataFrame({c: v["criteria"] for c, v in physiology["conditions"].items()})

In [ ]:
pd.read_csv(run.out / "physiology" / "physiologyChecks.csv")

In [ ]:
{c: round(v["lastBeatSpeedMPerS"], 4) for c, v in physiology["conditions"].items()}

In [ ]:
window = run.runStage("window")
window

In [ ]:
from efov.report import activationFigure
display(Image(str(activationFigure(run, run.folder("figures")))))